In [21]:
import sys
sys.path.append('..')

from api_client import AutoJWTClient
from pprint import pprint
from dotenv import dotenv_values
from datetime import date
import urllib3
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

In [22]:
client = AutoJWTClient(
    base_url="https://192.168.123.97:1000/api",
    login_url="/auth/token/",
    email="a@a.com",
    password="pars",
    verify=False
)

# index 

[How get counter list](#Add-New-Counter)

# Counter 

Counters listing

In [23]:
client.get('/counter/').json()

[{'name': 'pars', 'unit': 'Kez', 'count': 0, 'id': 1, 'split_type': 'weekly'},
 {'name': 'Nice', 'unit': 'Kez', 'count': 25, 'id': 2, 'split_type': 'weekly'},
 {'name': 'İhtiyaç',
  'unit': 'Kez',
  'count': -5,
  'id': 3,
  'split_type': 'daily'},
 {'name': 'İhtiyaç',
  'unit': 'Kez',
  'count': 33,
  'id': 4,
  'split_type': 'daily'},
 {'name': 'İhtiyaç',
  'unit': 'Kez',
  'count': 10,
  'id': 5,
  'split_type': 'daily'},
 {'name': 'İhtiyaç',
  'unit': 'Kez',
  'count': 8,
  'id': 6,
  'split_type': 'daily'},
 {'name': 'İhtiyaç',
  'unit': 'Tane',
  'count': 1,
  'id': 7,
  'split_type': 'daily'},
 {'name': 'İhtiyaç',
  'unit': 'Tane',
  'count': 23,
  'id': 8,
  'split_type': 'daily'},
 {'name': 'test', 'unit': '', 'count': 5, 'id': 9, 'split_type': 'daily'},
 {'name': 'test2', 'unit': '', 'count': 0, 'id': 10, 'split_type': 'weekly'},
 {'name': 'pars', 'unit': '1', 'count': 5, 'id': 11, 'split_type': 'daily'},
 {'name': 'İhtiyaç',
  'unit': 'Tane',
  'count': 1,
  'id': 12,
  'spl

---

# Add New Counter

In [24]:
#split types

DAILY = 'daily'
WEEKLY = 'weekly'
MONTHLY = 'monthly'

def add_new_counter(name: str, split_type:str, unit: str = 'single'):
    if response := client.post(
        path='/counter/',
        json={
            "unit": unit,
            "name": name,
            "split_type": split_type
        }
    ):
        pprint(response.json())
    else:
        print(response.text)
        

In [25]:
add_new_counter(name="test-daily-reset-counter", split_type=DAILY)

{'id': 18,
 'name': 'test-daily-reset-counter',
 'split_type': 'daily',
 'unit': 'single'}


In [26]:
add_new_counter(name="test-weekly-reset-counter", split_type=WEEKLY)

{'id': 19,
 'name': 'test-weekly-reset-counter',
 'split_type': 'weekly',
 'unit': 'single'}


In [27]:
add_new_counter(name="test-monthly-reset-counter", split_type=MONTHLY)

{'id': 20,
 'name': 'test-monthly-reset-counter',
 'split_type': 'monthly',
 'unit': 'single'}


---

# How counters tick

In [28]:
# Increasing Tick

counter_id = 1

# request
client.post(
    path='/counter/tick/',
    json={
        "counter": counter_id,
        "value": 1
    }
)



<Response [201]>

In [29]:
# Decreasing Tick

counter_id = 1

# request
client.post(
    path='/counter/tick/',
    json={
        "counter": counter_id,
        "value": -1
    }
)

<Response [201]>

In [30]:
# Custom DATE tick
# 
r = client.post(f'/counter/tick/', json={
    "counter": counter_id,
    "value": -1,
    "timestamp": "2025-06-10"
})
r

<Response [201]>

In [31]:
r.json()


{'id': 461,
 'timestamp': '2025-06-10T00:00:00+03:00',
 'value': -1,
 'counter': 1}

---

# Counter Base | LIST - CREATE

In [32]:
counter_id = 1

retrieve = client.get(path=f'/counter/{counter_id}/')

In [33]:
retrieve.json()

{'id': 1,
 'interval_count': 0,
 'name': 'pars',
 'unit': 'Kez',
 'split_type': 'weekly',
 'created_at': '2025-05-25T06:26:44.982121+03:00'}

---

# Update counter object 

In [34]:
counter_id = 1

retrieve = client.put(path=f'/counter/{counter_id}/', json={
    "name": "pars",
    "split_type": "weekly"
})

In [35]:
retrieve.text

'{"id":1,"interval_count":0,"name":"pars","unit":"Kez","split_type":"weekly","created_at":"2025-05-25T06:26:44.982121+03:00"}'

---

# get counter object entries form counter id 

In [36]:
counter_id = 1

retrieve = client.get(path=f'/counter/{counter_id}/entry/')

In [37]:
retrieve

<Response [200]>

In [43]:
[f"{index}, '->', {_}" for index, _ in enumerate(retrieve.json())]

["0, '->', {'date': '2025-06-09T00:00:00+03:00', 'count': -1}",
 "1, '->', {'date': '2025-08-04T00:00:00+03:00', 'count': 0}"]

---

# Delete counter interval entires

In [44]:
counter_id = 1
delete_index = 5

retrieve = client.delete(path=f'/counter/{counter_id}/entry/{delete_index}/')


In [46]:
retrieve.status_code # 404 | 400 | 204 

404

In [47]:
retrieve.text

'{}'

---

# Remove all entries from counter id 

In [50]:
counter_id = 1

retrieve = client.delete(path=f'/counter/{counter_id}/entry/delete-all/')

In [52]:
retrieve # ! this api does not remove counter object just delete counter entries 

<Response [204]>